In [69]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

In [70]:
# Plot
plot_directory = 'new_result/plot_set'

In [71]:
# colors = [
#     '#e6194B',
#     '#f58231',
#     '#9A6324',
#     '#911eb4',
#     '#3cb44b',
#     '#f032e6',
#     '#4363d8',
# ]

In [ ]:
colors = [
	'#FF0000',
	'#00FFFF',
	'#0000FF',
	'#00008B',
	'#ADD8E6',
	'#800080',
	'#7FFFD4',
	'#008000',
	'#FF00FF',
	'#FFC0CB',
	'#C0C0C0',
	'#FFA500',
	'#000000',
	'#800000',
]

In [ ]:
def load_json_file(file_path):
	try:
		with open(file_path, 'r') as file:
			data = json.load(file)
		return data
	
	except Exception as e:
		print(f"An error occurred while loading the JSON file: {e}")
		return None

In [ ]:
def extract_fpss(metric_list):
	return list(metric_list[list(metric_list.keys())[0]][0]['metric'].keys())

In [ ]:
def to_accuracy_vector(accuracy_result_seq, fpss, type='F1'):
	accuracy_vector = []
	for fps in fpss:
		accuracy_vector.append(accuracy_result_seq[fps][type])
	
	return accuracy_vector

In [ ]:
def plot_scatter(xs, ys, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	plt.scatter(xs, ys, c=colors[0], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [ ]:
def plot_scatter_label(xs, ys, labels, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	for i in range(len(xs)):
		plt.scatter(xs[i], ys[i], c=colors[labels[i]], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [ ]:
def filter_list_index(input_list, indices_to_remove):
	return [item for i, item in enumerate(input_list) if i not in indices_to_remove]

In [ ]:
def round_float_to_sigfigs(number, sigfigs):
	return round(number, sigfigs)

## Plot

In [80]:
omv_features = ["Left-Top", "Right-Top", "Left-Bottom", "Right-Bottom", "Object-Amount", "Confidence", "IOU", "Object Size"]

In [81]:
plot_filenames = sorted(os.listdir(plot_directory))
plot_video_names = sorted(list(set([f.split('_')[0] for f in plot_filenames])))

In [82]:
fpss = extract_fpss(load_json_file(os.path.join(plot_directory, plot_video_names[0] + "_Accuracy_Result.json")))
# fpss = ['2', '3', '5', '6', '10', '15']

In [ ]:
omv_videos = []
acc_videos = []

for v in plot_video_names:
	omv_dict = {}
	for fps in fpss:
		omv_dict[fps] = []
	acc_list = []

	accuracy_result = load_json_file(os.path.join(plot_directory, v + "_Accuracy_Result.json"))
	movement_result = load_json_file(os.path.join(plot_directory, v + "_Movement_Result.json"))

	for class_idx in list(accuracy_result.keys()):
		for i in range(len(accuracy_result[class_idx])):
			accuracy_vector = to_accuracy_vector(accuracy_result[class_idx][i]['metric'], fpss)
			acc_list.append(accuracy_vector)

			for fps in fpss:
				movement_vector = movement_result[class_idx][i]['movement'][fps]
				omv_dict[fps].append(movement_vector)
	
	omv_videos.append(omv_dict)
	acc_videos.append(acc_list)

In [ ]:
# Corr Plot Format 1

for i in range(len(plot_video_names)):
	video_name = plot_video_names[i]
	print(omv_videos[0][fpss[0]])
	for j in range(len(omv_videos[0][fpss[0]][0])):
		for k in range(len(fpss)):
			fps = fpss[k]
			
			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
			acc_fps = list(np.array(acc_videos[i])[:, k])

			# Remove Outliers
			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

			title = f'{video_name}; OMV Feature {j} ({omv_features[j]}); FPS: {fps}'

			correlation_matrix = np.corrcoef(omv_fps_clean, acc_fps_clean)
			correlation_coefficient = correlation_matrix[0, 1]
			print(f"{title} -> Corr: {round_float_to_sigfigs(correlation_coefficient, 3)}")
			# plot_scatter(omv_fps_clean, acc_fps_clean, 'OMV Feature', 'ACC', title)

		print("")

[[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0],

/Users/jiaxili/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:495: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis)
/Users/jiaxili/opt/anaconda3/lib/python3.8/site-packages/numpy/core/_methods.py:181: RuntimeWarning: invalid value encountered in true_divide
  ret = um.true_divide(
/Users/jiaxili/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2821: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/jiaxili/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2680: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)
/Users/jiaxili/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2680: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/Users/jiaxili/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2829: RuntimeWarning: invalid value encountered in true_divide
  c /= st


Video1; OMV Feature 4 (Object-Amount); FPS: 3 -> Corr: 0.355
Video1; OMV Feature 4 (Object-Amount); FPS: 5 -> Corr: 0.016
Video1; OMV Feature 4 (Object-Amount); FPS: 6 -> Corr: -0.112
Video1; OMV Feature 4 (Object-Amount); FPS: 10 -> Corr: -0.339
Video1; OMV Feature 4 (Object-Amount); FPS: 15 -> Corr: -0.361
Video1; OMV Feature 4 (Object-Amount); FPS: 30 -> Corr: nan

Video1; OMV Feature 5 (Confidence); FPS: 1 -> Corr: 0.252
Video1; OMV Feature 5 (Confidence); FPS: 2 -> Corr: 0.262
Video1; OMV Feature 5 (Confidence); FPS: 3 -> Corr: 0.3
Video1; OMV Feature 5 (Confidence); FPS: 5 -> Corr: 0.352
Video1; OMV Feature 5 (Confidence); FPS: 6 -> Corr: 0.425
Video1; OMV Feature 5 (Confidence); FPS: 10 -> Corr: 0.452
Video1; OMV Feature 5 (Confidence); FPS: 15 -> Corr: 0.466
Video1; OMV Feature 5 (Confidence); FPS: 30 -> Corr: nan

Video1; OMV Feature 6 (IOU); FPS: 1 -> Corr: 0.518
Video1; OMV Feature 6 (IOU); FPS: 2 -> Corr: 0.488
Video1; OMV Feature 6 (IOU); FPS: 3 -> Corr: 0.462
Video1; OMV

In [ ]:
# Corr Plot Format 2
np_result = np.zeros((len(fpss), len(omv_videos[0][fpss[0]][0])))

for k in range(len(fpss)):
	fps = fpss[k]
	print(f"FPS: {fps}")

	for j in range(len(omv_videos[0][fpss[0]][0])):
		print(f"OMV Feature {j} ({omv_features[j]})")
		corr_list = []

		for i in range(len(plot_video_names)):
			video_name = plot_video_names[i]
		
			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
			acc_fps = list(np.array(acc_videos[i])[:, k])

			# Remove Outliers
			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

			correlation_matrix = np.corrcoef(omv_fps_clean, acc_fps_clean)
			correlation_coefficient = correlation_matrix[0, 1]
			corr_list.append(correlation_coefficient)

		print(f'Average Corr Among All {len(plot_video_names)} Videos {round_float_to_sigfigs(np.average(np.array(corr_list)), 3)}')
		np_result[k][j] = round_float_to_sigfigs(np.average(np.array(corr_list)), 3)

		print("")

FPS: 1
OMV Feature 0 (Left-Top)
Average Corr Among All 10 Videos nan

OMV Feature 1 (Right-Top)
Average Corr Among All 10 Videos nan

OMV Feature 2 (Left-Bottom)
Average Corr Among All 10 Videos nan

OMV Feature 3 (Right-Bottom)
Average Corr Among All 10 Videos nan

OMV Feature 4 (Object-Amount)
Average Corr Among All 10 Videos nan

OMV Feature 5 (Confidence)
Average Corr Among All 10 Videos nan

OMV Feature 6 (IOU)
Average Corr Among All 10 Videos nan

OMV Feature 7 (Object Size)
Average Corr Among All 10 Videos nan

FPS: 2
OMV Feature 0 (Left-Top)
Average Corr Among All 10 Videos -0.064

OMV Feature 1 (Right-Top)
Average Corr Among All 10 Videos -0.071

OMV Feature 2 (Left-Bottom)
Average Corr Among All 10 Videos -0.066

OMV Feature 3 (Right-Bottom)
Average Corr Among All 10 Videos -0.071

OMV Feature 4 (Object-Amount)
Average Corr Among All 10 Videos 0.385

OMV Feature 5 (Confidence)
Average Corr Among All 10 Videos 0.128

OMV Feature 6 (IOU)
Average Corr Among All 10 Videos 0.288



In [86]:
print(np_result.tolist())

[[nan, nan, nan, nan, nan, nan, nan, nan], [-0.064, -0.071, -0.066, -0.071, 0.385, 0.128, 0.288, 0.081], [-0.046, -0.044, -0.05, -0.046, 0.395, 0.156, 0.281, 0.034], [0.041, 0.045, 0.038, 0.043, 0.314, 0.156, 0.468, 0.075], [0.085, 0.097, 0.082, 0.095, 0.287, 0.188, 0.477, 0.08], [0.094, 0.107, 0.091, 0.104, 0.198, 0.185, 0.547, 0.047], [0.134, 0.146, 0.128, 0.141, 0.176, 0.193, 0.55, 0.039], [nan, nan, nan, nan, nan, nan, nan, nan]]
